In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
url = "https://raw.githubusercontent.com/Avik-Jain/100-Days-Of-ML-Code/master/datasets/Social_Network_Ads.csv"
df = pd.read_csv(url)
df.head()

## **Understanding the Dataset**

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
# Check duplicates before dropping User ID
print("Duplicates with User ID:", df.duplicated().sum())
df.drop(columns=["User ID"], inplace=True)

In [ ]:
df.sample(5, random_state=42)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])
print(dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
df["Gender"].dtype

In [ ]:
df.sample(5, random_state=42)

In [ ]:
df.isnull().sum()

In [ ]:
df.describe().T

In [ ]:
df.duplicated().sum()

In [ ]:
display(df["Purchased"].value_counts().sort_index().rename(
    index={0: "Did not purchase (0)", 1: "Purchased (1)"}
))

## **EDA**

In [ ]:
plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x="Purchased", hue="Purchased")
if ax.get_legend() is not None:
    ax.get_legend().remove()
ax.set_title("Customer Purchase Distribution")
ax.set_xlabel("Purchased (0 = No, 1 = Yes)")
ax.set_ylabel("Number of customers")
plt.tight_layout()
plt.show()

print((df["Purchased"].value_counts(normalize=True).sort_index() * 100).round(1))

In [ ]:
# Age distribution
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="Age", bins=20, kde=True)
plt.title("Distribution of Customer Age")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Salary Distribution
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="EstimatedSalary", bins=25, kde=True)
plt.title("Distribution of Estimated Salary")
plt.xlabel("Estimated salary")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, x="Age", y="EstimatedSalary",
    hue="Purchased", style="Purchased", alpha=0.8
)
plt.title("Age vs Estimated Salary (colored by purchase outcome)")
plt.tight_layout()
plt.show()

## **Correlation**

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm",
            center=0, fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## **Training**

In [ ]:
X = df[["Gender", "Age", "EstimatedSalary"]]
y = df["Purchased"]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

# Make copies, then scale only Age and EstimatedSalary (fit on train only)
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

numeric_columns = ["Age", "EstimatedSalary"]

X_train_scaled[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])
X_test_scaled[numeric_columns] = scaler.transform(X_test[numeric_columns])

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

In [ ]:
y_pred = knn.predict(X_test_scaled)
y_pred

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy (K=5): {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (K=5)")
plt.tight_layout()
plt.show()

In [ ]:
# Find optimal K using cross-validation on the training set (avoiding test-set leakage)
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

k_range = range(1, 21, 2)  # Odd values avoid distance ties in binary classification
cv_means = []

for k in k_range:
    pipe = Pipeline([
        ("scaler", ColumnTransformer([("num", StandardScaler(), numeric_columns)], remainder="passthrough")),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    cv_means.append(scores.mean())

best_k = list(k_range)[int(np.argmax(cv_means))]
print(f"Best K (5-fold CV): {best_k} | Mean CV Accuracy: {max(cv_means):.2%}")

model = KNeighborsClassifier(n_neighbors=best_k)
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)
score = accuracy_score(y_test, predictions)

print(f"Test Accuracy (K = {best_k}): {score:.2%}")

## **Prediction**

In [ ]:
new_person = pd.DataFrame({
    "Age": [30],
    "EstimatedSalary": [60000],
    "Gender": [1]  # 1 = Male, 0 = Female
})

# Keep the same column order used during training
new_person = new_person[X.columns]

new_person_scaled = new_person.copy()
new_person_scaled[numeric_columns] = scaler.transform(new_person[numeric_columns])

prediction = model.predict(new_person_scaled)[0]

if prediction == 1:
    print("Prediction: This person may purchase the product.")
else:
    print("Prediction: This person may not purchase the product.")